# Task 3: Data Cleaning

## Objective

The objective of this project is to identify and resolve data quality
issues in a deliberately messy employee dataset.

The cleaning process includes:

- Data quality assessment
- Missing value handling
- Duplicate removal
- Data standardization
- Data type correction
- Outlier detection using the IQR method
- Before vs After comparison
- Final data validation
- Exporting the cleaned dataset

### Import Libraries & Load the Dataset

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [36]:
df = pd.read_csv("Messy_Employee_dataset.csv")
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [37]:
df.shape

(1020, 12)

In [39]:
df.columns.tolist()

['Employee_ID',
 'First_Name',
 'Last_Name',
 'Age',
 'Department_Region',
 'Status',
 'Join_Date',
 'Salary',
 'Email',
 'Phone',
 'Performance_Score',
 'Remote_Work']

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   object 
 1   First_Name         1020 non-null   object 
 2   Last_Name          1020 non-null   object 
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   object 
 5   Status             1020 non-null   object 
 6   Join_Date          1020 non-null   object 
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   object 
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   object 
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(8)
memory usage: 88.8+ KB


### 1. Initial Data Quality Check

Before cleaning the dataset, we inspect the data to identify:

- Missing values
- Duplicate records
- Incorrect data types
- Inconsistent formatting
- Invalid numerical values
- Potential outliers

### Missing Values

In [41]:
missing_values = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

missing_values

,Missing Count,Missing Percentage
Employee_ID,0,0.00
First_Name,0,0.00
Last_Name,0,0.00
Age,211,20.69
Department_Region,0,0.00
Status,0,0.00
Join_Date,0,0.00
Salary,24,2.35
Email,0,0.00
Phone,0,0.00


### Duplicate Count

In [42]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


### Save Before-Cleaning Statistics

In [43]:
before_rows = len(df)
before_duplicates = df.duplicated().sum()
before_missing = df.isnull().sum().sum()

print("Rows before cleaning:", before_rows)
print("Duplicates before cleaning:", before_duplicates)
print("Missing values before cleaning:", before_missing)

Rows before cleaning: 1020
Duplicates before cleaning: 0
Missing values before cleaning: 235


### Check Data Types

In [44]:
dtype_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

dtype_report

,Column,Data Type
0,Employee_ID,object
1,First_Name,object
2,Last_Name,object
3,Age,float64
4,Department_Region,object
5,Status,object
6,Join_Date,object
7,Salary,float64
8,Email,object
9,Phone,int64


### Check Numerical Columns

In [45]:
df.describe()

,Age,Salary,Phone
count,809.000000,996.000000,1.020000e+03
mean,32.484549,85155.056396,-4.942253e+09
std,5.656860,19873.727918,2.817326e+09
min,25.000000,50047.320000,-9.994973e+09
25%,25.000000,68392.487500,-7.341992e+09
50%,30.000000,85547.870000,-4.943997e+09
75%,40.000000,100974.027500,-2.520391e+09
max,40.000000,119971.650000,-3.896086e+06


### Check Categorical Values

In [46]:
for column in df.select_dtypes(include="object").columns:
    print("\n" + "=" * 50)
    print(column)
    print(df[column].value_counts(dropna=False).head(20))


Employee_ID
Employee_ID
EMP2019    1
EMP1000    1
EMP1001    1
EMP1002    1
EMP1003    1
EMP1004    1
EMP1005    1
EMP1006    1
EMP1007    1
EMP1008    1
EMP1009    1
EMP1010    1
EMP2003    1
EMP2002    1
EMP2001    1
EMP2000    1
EMP1999    1
EMP1998    1
EMP1997    1
EMP1996    1
Name: count, dtype: int64

First_Name
First_Name
Frank      142
Grace      140
Eva        136
Bob        133
Charlie    125
Alice      117
David      116
Heidi      111
Name: count, dtype: int64

Last_Name
Last_Name
Brown       148
Garcia      136
Smith       136
Jones       131
Davis       120
Johnson     118
Miller      116
Williams    115
Name: count, dtype: int64

Department_Region
Department_Region
HR-Florida               41
DevOps-California        35
Sales-Nevada             35
Admin-Nevada             34
Admin-California         34
DevOps-Florida           34
Finance-Illinois         33
DevOps-New York          33
Sales-Florida            33
HR-New York              33
DevOps-Illinois          33


### Clean Names
### Standardizing Name Fields

The First_Name and Last_Name columns are cleaned by:

- Removing leading/trailing spaces
- Converting names to consistent title case
- Replacing missing names with "Unknown"

In [47]:
df["First_Name"] = (
    df["First_Name"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

df["Last_Name"] = (
    df["Last_Name"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

In [48]:
df[["First_Name", "Last_Name"]].head()

,First_Name,Last_Name
0,Bob,Davis
1,Bob,Brown
2,Alice,Jones
3,Eva,Davis
4,Frank,Williams


### Clean Age

In [49]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

In [50]:
print("Missing Age values:", df["Age"].isnull().sum())

Missing Age values: 211


In [52]:
#Fill missing Age using median
df["Age"] = df["Age"].fillna(df["Age"].median())

In [53]:
df["Age"].isnull().sum()

np.int64(0)

### Check Age Outliers

In [54]:
Q1 = df["Age"].quantile(0.25)
Q3 = df["Age"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

age_outliers = df[
    (df["Age"] < lower_bound) |
    (df["Age"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of Age outliers:", len(age_outliers))

Lower bound: 22.5
Upper bound: 42.5
Number of Age outliers: 0


In [55]:
age_outliers[["Employee_ID", "Age"]]

,Employee_ID,Age


### Handle Impossible Age Values

In [57]:
invalid_age_count = ((df["Age"] < 18) | (df["Age"] > 100)).sum()

print("Invalid age records:", invalid_age_count)

Invalid age records: 0


In [58]:
df = df[(df["Age"] >= 18) & (df["Age"] <= 100)]

In [59]:
df

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,30.0,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,30.0,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,EMP2015,David,Miller,30.0,HR-California,Active,8/19/2023,NaN,david.miller@example.com,-3546212759,Good,True
1016,EMP2016,David,Johnson,30.0,Cloud Tech-Texas,Inactive,11/7/2021,100215.06,david.johnson@example.com,-2508261122,Good,True
1017,EMP2017,Charlie,Williams,40.0,Finance-New York,Active,10/4/2023,114587.11,charlie.williams@example.com,-1261632487,Average,False
1018,EMP2018,Alice,Garcia,30.0,HR-Florida,Inactive,12/16/2024,71318.79,alice.garcia@example.com,-8995729892,Good,True


### Clean Department and Region
### Standardizing Department and Region

The Department_Region column contains two pieces of information:
Department and Region.

The column is split into two separate fields to improve structure
and make the dataset easier to analyze.

In [60]:
df["Department_Region"] = (
    df["Department_Region"]
    .fillna("Unknown-Unknown")
    .astype(str)
    .str.strip()
)

In [61]:
df[["Department", "Region"]] = df["Department_Region"].str.split(
    "-", n=1, expand=True
)

In [62]:
df["Department"] = (
    df["Department"]
    .fillna("Unknown")
    .str.strip()
    .str.title()
)

df["Region"] = (
    df["Region"]
    .fillna("Unknown")
    .str.strip()
    .str.title()
)

In [63]:
df[["Department_Region", "Department", "Region"]].head()

,Department_Region,Department,Region
0,DevOps-California,Devops,California
1,Finance-Texas,Finance,Texas
2,Admin-Nevada,Admin,Nevada
3,Admin-Nevada,Admin,Nevada
4,Cloud Tech-Florida,Cloud Tech,Florida


### Clean Status

In [64]:
df["Status"] = (
    df["Status"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

In [65]:
df["Status"].value_counts(dropna=False)

Status
Pending     356
Active      352
Inactive    312
Name: count, dtype: int64

### Clean Performance Score

In [66]:
df["Performance_Score"].value_counts(dropna=False)

Performance_Score
Good         270
Average      267
Excellent    267
Poor         216
Name: count, dtype: int64

### Clean Email

In [67]:
df["Email"] = (
    df["Email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [68]:
df["Email"].head()

0         bob.davis@example.com
1         bob.brown@example.com
2       alice.jones@example.com
3         eva.davis@example.com
4    frank.williams@example.com
Name: Email, dtype: string

### Clean Phone

In [69]:
df["Phone"] = (
    df["Phone"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
)

In [70]:
df["Phone"].head()

0    1651623197
1    1898471390
2    5596363211
3    3476490784
4    1586734256
Name: Phone, dtype: string

### Check Invalid Phone Numbers

In [71]:
phone_length = df["Phone"].str.len()

print("Phone numbers with fewer than 10 digits:",
      (phone_length < 10).sum())

print("Phone numbers with more than 10 digits:",
      (phone_length > 10).sum())

Phone numbers with fewer than 10 digits: 92
Phone numbers with more than 10 digits: 0


In [72]:
df.loc[
    (phone_length < 10) | (phone_length > 10),
    ["Employee_ID", "Phone"]
]

,Employee_ID,Phone
43,EMP1043,164597793
45,EMP1045,61027768
47,EMP1047,310303722
58,EMP1058,54522800
68,EMP1068,846684549
84,EMP1084,185106793
100,EMP1100,794200748
105,EMP1105,26876810
107,EMP1107,175370303
112,EMP1112,894870604


### Clean Join Date
### Date Type Correction

Join_Date contains date values that may use inconsistent formatting.

The column is converted to datetime so that dates can be reliably
sorted, filtered, compared and analyzed.

In [73]:
df["Join_Date"] = pd.to_datetime(
    df["Join_Date"],
    errors="coerce"
)

In [74]:
print(df["Join_Date"].dtype)

datetime64[ns]


### Check Invalid Dates

In [76]:
print("Invalid/missing dates:", df["Join_Date"].isnull().sum())

Invalid/missing dates: 0


In [77]:
df[df["Join_Date"].isnull()]

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region


### Clean Salary

In [78]:
df["Salary"] = pd.to_numeric(
    df["Salary"],
    errors="coerce"
)

In [79]:
print("Missing Salary values:", df["Salary"].isnull().sum())

Missing Salary values: 24


### Fill Missing Salary

In [80]:
df["Salary"] = df["Salary"].fillna(
    df["Salary"].median()
)

In [81]:
print("Missing Salary after cleaning:",
      df["Salary"].isnull().sum())

Missing Salary after cleaning: 0


### Salary Outlier Detection

In [82]:
Q1 = df["Salary"].quantile(0.25)
Q3 = df["Salary"].quantile(0.75)

IQR = Q3 - Q1

lower_salary = Q1 - 1.5 * IQR
upper_salary = Q3 + 1.5 * IQR

salary_outliers = df[
    (df["Salary"] < lower_salary) |
    (df["Salary"] > upper_salary)
]

print("Lower salary boundary:", lower_salary)
print("Upper salary boundary:", upper_salary)
print("Salary outliers:", len(salary_outliers))

Lower salary boundary: 21469.087499999987
Upper salary boundary: 147714.80750000002
Salary outliers: 0


In [83]:
salary_outliers[["Employee_ID", "Salary"]]

,Employee_ID,Salary


### Remote Work

In [84]:
df["Remote_Work"].value_counts(dropna=False)

Remote_Work
True     513
False    507
Name: count, dtype: int64

In [85]:
df["Remote_Work"] = df["Remote_Work"].astype("boolean")

In [86]:
df["Remote_Work"].dtype

BooleanDtype

### Remove Duplicate Rows

In [87]:
duplicate_count_after_cleaning = df.duplicated().sum()

print(
    "Duplicate rows before removal:",
    duplicate_count_after_cleaning
)

Duplicate rows before removal: 0


In [91]:
df = df.drop_duplicates().reset_index(drop=True)

In [92]:
print("Duplicate rows after removal:", df.duplicated().sum())

Duplicate rows after removal: 0


### Document Duplicate Removal

In [93]:
duplicates_removed = (
    duplicate_count_after_cleaning -
    df.duplicated().sum()
)

print("Total duplicate rows removed:", duplicates_removed)

Total duplicate rows removed: 0


### Final Missing Value Check

In [94]:
final_missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

final_missing

,Missing Count,Missing Percentage
Employee_ID,0,0.0
First_Name,0,0.0
Last_Name,0,0.0
Age,0,0.0
Department_Region,0,0.0
Status,0,0.0
Join_Date,0,0.0
Salary,0,0.0
Email,0,0.0
Phone,0,0.0


### Final Data Types

In [95]:
df.dtypes

Employee_ID                  object
First_Name                   object
Last_Name                    object
Age                         float64
Department_Region            object
Status                       object
Join_Date            datetime64[ns]
Salary                      float64
Email                string[python]
Phone                string[python]
Performance_Score            object
Remote_Work                 boolean
Department                   object
Region                       object
dtype: object

### Final Data Quality Report

In [96]:
final_quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing %": (df.isnull().mean() * 100).round(2).values,
    "Unique Values": df.nunique().values
})

final_quality_report

,Column,Data Type,Missing Values,Missing %,Unique Values
0,Employee_ID,object,0,0.0,1020
1,First_Name,object,0,0.0,8
2,Last_Name,object,0,0.0,8
3,Age,float64,0,0.0,4
4,Department_Region,object,0,0.0,36
5,Status,object,0,0.0,3
6,Join_Date,datetime64[ns],0,0.0,760
7,Salary,float64,0,0.0,979
8,Email,string,0,0.0,64
9,Phone,string,0,0.0,1020


### Before vs After

In [97]:
after_rows = len(df)
after_duplicates = df.duplicated().sum()
after_missing = df.isnull().sum().sum()

In [98]:
comparison = pd.DataFrame({
    "Metric": [
        "Row Count",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        before_rows,
        before_missing,
        before_duplicates
    ],
    "After Cleaning": [
        after_rows,
        after_missing,
        after_duplicates
    ]
})

comparison

,Metric,Before Cleaning,After Cleaning
0,Row Count,1020,1020
1,Missing Values,235,0
2,Duplicate Rows,0,0


### Final Dataset Preview

In [99]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,2021-04-02,59767.65,bob.davis@example.com,1651623197,Average,True,Devops,California
1,EMP1001,Bob,Brown,30.0,Finance-Texas,Active,2020-07-10,65304.66,bob.brown@example.com,1898471390,Excellent,True,Finance,Texas
2,EMP1002,Alice,Jones,30.0,Admin-Nevada,Pending,2023-12-07,88145.90,alice.jones@example.com,5596363211,Good,True,Admin,Nevada
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,2021-11-27,69450.99,eva.davis@example.com,3476490784,Good,True,Admin,Nevada
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,2022-01-05,109324.61,frank.williams@example.com,1586734256,Poor,False,Cloud Tech,Florida


### Final Dataset Shape

In [100]:
print("Final rows:", df.shape[0])
print("Final columns:", df.shape[1])

Final rows: 1020
Final columns: 14


### Final Validation

In [101]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Rows:", len(df))
print("Columns:", len(df.columns))

Missing values: 0
Duplicate rows: 0
Rows: 1020
Columns: 14


### Save Clean Dataset

In [104]:
df.to_csv(
    "Messy_Employee_dataset_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


### Save Quality Report

In [105]:
final_quality_report.to_csv(
    "Messy_Employee_dataset_cleaned.csv",
    index=False
)

print("Data quality report saved successfully.")

Data quality report saved successfully.


### Final Conclusion

The Messy Employee Dataset was systematically cleaned and transformed
into an analysis-ready dataset.

The following data quality issues were addressed:

1. Missing values were identified and appropriately treated.
2. Duplicate records were detected and removed.
3. Text fields were standardized.
4. Department and Region were separated from the compound field.
5. Join_Date was converted to datetime format.
6. Salary values were converted to numeric format.
7. Potential numerical outliers were identified using the IQR method.
8. Phone numbers were standardized as text.
9. Remote_Work was converted to Boolean format.
10. A before-vs-after data quality comparison was performed.

The final cleaned dataset was exported as a CSV file.